# Almond Scala - запуск SPARK сессии

## Подключение библиотек Spark

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-mllib:4.1.1` // если нужен ML
import $ivy.`org.apache.spark::spark-hive:4.1.1`

import $ivy.$
import $ivy.$
import $ivy.$

## Spark - сессия

In [2]:
import scala.io.Source._
import scala.sys.process._

import scala.io.Source._
import scala.sys.process._

In [3]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}
// Подавить сообщения типа INFO при загрзке библиотек Scala
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)
// Spark - сессия
val spark = SparkSession.builder()
  .appName("Spark-notebook")
  .master("local[2]")
  .config("spark.sql.warehouse.dir", "spark-warehouse")
  .config("spark.sql.hive.metastore.version", "2.3.10") // Укажите версию вашего метастора Hive
  .config("spark.ui.port", "4040")
  .enableHiveSupport()          
  .getOrCreate()

import spark.implicits._ // активация методов Saprk для объектов Scala

val sc = spark.sparkContext // Spatk - context
spark.sparkContext.setLogLevel("WARN") // Уровень сообщений - WARNING

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 20:59:05 WARN Utils: Your hostname, user-PC, resolves to a loopback address: 127.0.1.1; using 192.168.88.252 instead (on interface eno1)
26/04/30 20:59:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/30 20:59:05 INFO SparkContext: Running Spark version 4.1.1
26/04/30 20:59:05 INFO SparkContext: OS info Linux, 7.0.0-15-generic, amd64
26/04/30 20:59:05 INFO SparkContext: Java version 17.0.18+8-Ubuntu-1
26/04/30 20:59:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 20:59:05 INFO ResourceUtils: ==============================================================
26/04/30 20:59:05 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/30 20:59:05 INFO ResourceUtils: ==============================================================
26/04/30 20:59:06 INFO BlockManagerMasterEn

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@309cbd
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@64b2f3b1

In [4]:
println(
s"""Spark version: ${spark.version}
Scala version: ${scala.util.Properties.versionString}
App Name: ${spark.sparkContext.appName}
Master: ${spark.sparkContext.master}
Application Id: ${spark.sparkContext.applicationId}
Spark UI: ${spark.sparkContext.uiWebUrl.getOrElse("нет")}"""
)

Spark version: 4.1.1
Scala version: version 2.13.17
App Name: Spark-notebook
Master: local[2]
Application Id: local-1777571946333
Spark UI: http://192.168.88.252:4040


In [5]:
val df = Seq((1, "Spark", 25), (2, "Hive", 30), (3, "Scala", 35)).toDF("id", "name", "count")
df.show() // Показать содержимое DataFrame

+---+-----+-----+
| id| name|count|
+---+-----+-----+
|  1|Spark|   25|
|  2| Hive|   30|
|  3|Scala|   35|
+---+-----+-----+



df: org.apache.spark.sql.package.DataFrame = [id: int, name: string ... 1 more field]

In [6]:
spark.sql("drop database if exists my_db cascade")
spark.sql("create database my_db")

26/04/30 20:59:11 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/04/30 20:59:11 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore user@127.0.1.1
26/04/30 20:59:12 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/04/30 20:59:12 WARN ObjectStore: Failed to get database my_db, returning NoSuchObjectException
26/04/30 20:59:12 WARN ObjectStore: Failed to get database my_db, returning NoSuchObjectException
26/04/30 20:59:12 WARN ObjectStore: Failed to get database my_db, returning NoSuchObjectException


res6_0: org.apache.spark.sql.package.DataFrame = []
res6_1: org.apache.spark.sql.package.DataFrame = []

In [7]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|    my_db|
+---------+



In [8]:
spark.sql("show tables from my_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [9]:
df.write.format("parquet").mode("overwrite").saveAsTable("my_db.test_table")

26/04/30 20:59:14 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/30 20:59:14 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


In [10]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|    my_db|
+---------+



In [11]:
val dfRead = spark.table("my_db.test_table")

dfRead: org.apache.spark.sql.package.DataFrame = [id: int, name: string ... 1 more field]

In [12]:
dfRead.show()

+---+-----+-----+
| id| name|count|
+---+-----+-----+
|  1|Spark|   25|
|  2| Hive|   30|
|  3|Scala|   35|
+---+-----+-----+



In [13]:
spark.stop()